In [1]:
from Run_Designs import design_experiments, design_initial_experiments
from methods.DataPrep import dataprep
import numpy as np
import pandas as pd

1. C - List of number of categroies corresponding to each categorical variable
2. Nc - Number of categorical variables
3. Nx - Continuous variables
4. h - categorical
5. x - continuous
6. z - [h, x]

7. nDim - Nc + Nx
8. bounds - Lower and Upper bounds of both Catergorical variables and continuous variables

9. n_iter - Number of iterations to run the algorithm
10. initN - Number of intial data points
11. batch_size (b) - Number of experiments to be generated in each iteration

12. acq_type - Acquisition Type
13. Meas_Noise - Measurement Noise in the target 
14. Trade-off : Explorarion exploitation balance

In [2]:
Stock_solid = pd.read_excel('CarbonSourceInfo.xlsx', 'Stocks_solid')
Stock_liquid = pd.read_excel('CarbonSourceInfo.xlsx', 'Stocks_liquid')

Carbon_Names = Stock_solid['Carbon Source'].values.tolist()
Carbon_Names.append(Stock_liquid['Carbon Source'][1])
Carbon_Names.append(Stock_liquid['Carbon Source'][2])

Carbon_Ub = [50]*17 #[g/L]
Carbon_Ub.append(10) # Glycerol [mL/L]
Carbon_Ub.append(50) #Ethanol [mL/L]

In [3]:
Mol_List = ['KPhaffii_HSA'] #, 'KPhaffii_RBDJ', 'Kphaffii_Trastuzumab'
Data_Molecules = {}
for m in Mol_List:
    Data_Molecules[m] = pd.read_excel("KPhaffii_Experiments.xlsx", m).iloc[:,1:]
    for i in range(Data_Molecules[m].shape[0]):
        Data_Molecules[m].iloc[i,0] = Carbon_Names.index(Data_Molecules[m].iloc[i,0])
        Data_Molecules[m].iloc[i,1] = Data_Molecules[m].iloc[i,1]/Carbon_Ub[Data_Molecules[m].iloc[i,0]]
        Data_Molecules[m].iloc[i,2:4] = Data_Molecules[m].iloc[i,2:4]/10
        
data_modified_np = {}
result_modified_np = {}
cnt_m = 0
for m in Mol_List:
    data_modified = []
    result_modified = []
    data_init = Data_Molecules[m].iloc[:,0:4].values.astype('float64')
    result_init = Data_Molecules[m]['SP [mg/L/OD600]'].values.reshape(-1,1).astype('float64')
    for nc in range(data_init.shape[0]):
#         if data_init[nc, 1] == 0:
#             temp = np.concatenate((np.arange(0, 19).reshape(-1, 1), np.zeros((19, 1)),
#                                    np.multiply(data_init[nc, 2:], np.ones((19, 2)))), axis=1)
#             temp_res = result_init[nc] * np.ones((19, 1))
#             data_modified + temp.tolist()
#             result_modified + temp_res.tolist()
#         else:
        data_modified.append(data_init[nc, :])
        result_modified.append(result_init[nc])
    
    temp1 = np.array(data_modified)
    data_modified_np[m] = np.concatenate((temp1, cnt_m*np.ones((temp1.shape[0], 1))), axis =1)
    result_modified_np[m] = np.array(result_modified).reshape(-1,1)
    
    cnt_m = cnt_m+1

In [4]:
data = data_modified_np['KPhaffii_HSA']
result = result_modified_np['KPhaffii_HSA']
Nround = 2
path = '../Exp/RBDJ/'


for r in range(Nround):
    data_modified = []
    result_modified = []
    dat_read = pd.read_csv("./Round"+str(r)+'/Reconstructed_Round'+str(r)+"_Coreg.csv").iloc[:,1:-1].values #
    result_new = pd.read_csv("./Round"+str(r)+ '/Pred_Round'+str(r) +"_Coreg_SP.csv").iloc[:,1].values.reshape(-1,1)
        #pd.read_csv(path + "Round0_Result_Summary_final.csv")['Specific Productivity'].values.reshape(-1,1)
    
    for nc in range(dat_read.shape[0]):
#         if dat_read[nc, 1] == 0:
#             temp = np.concatenate((np.arange(0, 19).reshape(-1, 1), np.zeros((19, 1)),
#                                    np.multiply(dat_read[nc, 2:], np.ones((19, 2)))), axis=1)
#             temp_res = result_new[nc] * np.ones((19, 1))
#             data_modified = data_modified + temp.tolist()
#             result_modified = result_modified + temp_res.tolist()
#         else:
        data_modified.append(dat_read[nc, :])
        result_modified.append(result_new[nc])
    
    temp1 = np.array(data_modified)
    data_new = np.concatenate((temp1, 1 * np.ones((temp1.shape[0], 1))), axis =1)
    result_new_np = np.array(result_modified).reshape(-1,1)
    
    
    data = np.concatenate((data, data_new), axis = 0)
    result = np.concatenate((result, result_new_np), axis = 0)

In [5]:
print(data.shape)
print(result.shape)

(106, 5)
(106, 1)


In [6]:
import pickle 
fnl = "./Round1/1_ModelBackground_Coreg.pkl" #_updated
with open(fnl, "rb") as fp:
    ModelBackground = pickle.load(fp)

In [7]:
key = 'Coreg'
data_param_Coreg = ModelBackground['data_param']
data_mod = dataprep(data, data_param_Coreg, key)

In [8]:
method = 'constant_liar'
batch_size = 11
output_file_name = './Round2/Round2_Coreg'
background_model_parameter = './Round1/1_ModelBackground_Coreg.pkl' #_updated
new_model_parameter = './Round2/2_ModelBackground_Coreg.pkl'

z_next, gp_actual = design_experiments(data_mod, result,batch_size, method,background_model_parameter, 
                            new_model_parameter , output_file_name)

[[13]
 [11]
 [11]
 [18]
 [ 1]
 [ 0]
 [ 3]
 [ 9]
 [14]
 [ 8]
 [14]]
[array([1.70922094, 1.56997308, 1.        , 1.44341141, 1.        ,
       1.        , 1.        , 1.        , 1.94270256, 1.8118946 ,
       1.        , 2.61762831, 1.        , 1.60349398, 2.52537349,
       1.        , 1.        , 1.        , 1.20133095])]
  ICM.                                           |               value  |  constraints  |  priors
  sum.MixtureViaSumAndProduct.mix                |  0.5495384437354309  |      +ve      |        
  sum.MixtureViaSumAndProduct.Mat52.lengthscale  |                (3,)  |      +ve      |        
  sum.white.variance                             |                 0.1  |   +ve fixed   |        
  B.W                                            |              (2, 2)  |               |        
  B.kappa                                        |                (2,)  |      +ve      |        
0
(5000, 4)
1
[2.         0.02750594 0.07154459 0.20342632 1.        ]
1
(5000, 4)
1
[

In [9]:
z_next

array([[2.00000000e+00, 2.75059443e-02, 7.15445945e-02, 2.03426315e-01,
        1.00000000e+00],
       [5.00000000e+00, 2.27730621e-03, 3.01457147e-01, 1.12690019e-01,
        1.00000000e+00],
       [1.20000000e+01, 1.64388349e-02, 2.56090325e-02, 5.77830626e-01,
        1.00000000e+00],
       [8.00000000e+00, 9.79843125e-01, 9.53864305e-02, 4.43393624e-01,
        1.00000000e+00],
       [4.00000000e+00, 1.99226161e-02, 9.11602833e-03, 7.62876141e-01,
        1.00000000e+00],
       [9.00000000e+00, 3.98994454e-01, 8.33142314e-02, 5.33843335e-01,
        1.00000000e+00],
       [1.30000000e+01, 2.10311816e-03, 5.98552629e-01, 6.11361223e-01,
        1.00000000e+00],
       [8.00000000e+00, 2.13822500e-01, 7.51109696e-01, 4.68471041e-01,
        1.00000000e+00],
       [0.00000000e+00, 7.97886142e-01, 9.23953853e-01, 7.93710880e-01,
        1.00000000e+00],
       [1.50000000e+01, 9.96322346e-01, 3.46697636e-01, 8.80599163e-01,
        1.00000000e+00],
       [1.60000000e+01, 4.6355

In [ ]:
# import pickle
# fnl = "./Round0/0_ModelBackground_Coreg.pkl"
# with open(fnl, "rb") as fp:
#     ModelBackground = pickle.load(fp)
    
# ModelBackground['Categorical_dist_param']['ht_batch_list'] = dat_read[:,0:1].tolist()

# with open("./Round0/0_ModelBackground_Coreg_updated.pkl", 'wb') as output:
#     # Pickle dictionary using protocol 0.
#     pickle.dump(ModelBackground, output)

In [ ]:
Carbon_Names[17]